# Days 5-6: Baseline with a frozen backbone
**Use a GPU runtime** (Runtime > Change runtime type > T4 GPU).

Trains only a new 3-class head on top of ImageNet ResNet18 (every pretrained layer frozen). This is your **"before" number**: Phase 2 will unfreeze layers and be compared against it.

**The test set is not used anywhere in this notebook.** It stays untouched until Day 9, after all tuning is finished. Everything you decide today (which epoch is best, whether it's good enough) is decided on the validation set.

It saves the best checkpoint (by validation accuracy) to Google Drive so a runtime reset can't lose it. Expect roughly 1.5 to 2 minutes per epoch (Day 4 measured data loading at about 88 images/s, which is the bottleneck), so about 20 to 25 minutes for the default 12 epochs.

## 1. Setup

In [ ]:
import os, time, random, zipfile, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torchvision
from torch import nn
from torchvision import datasets, transforms

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU: switch the runtime to a T4 GPU")

SAVE_TO_DRIVE = True          # keep checkpoints on Drive so a runtime reset can't lose them
CKPT_DIR = "/content/ckpt"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/maize_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

ROOT = "/content/maize_split"
if not os.path.isdir(ROOT + "/train"):
    z = "/content/drive/MyDrive/maize_split.zip"
    if not os.path.exists(z):
        from google.colab import drive as _d
        _d.mount("/content/drive")
    assert os.path.exists(z), "No /content/maize_split and no maize_split.zip on Drive: re-run Day 3 (and its optional Drive cell)."
    os.makedirs(ROOT, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(ROOT)
print("data folders:", sorted(os.listdir(ROOT)), "| checkpoints go to:", CKPT_DIR)


## 2. Data (same transforms as Day 4; the test set is deliberately not loaded)

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_ds = datasets.ImageFolder(ROOT + "/train", train_tf)
val_ds = datasets.ImageFolder(ROOT + "/val", eval_tf)
classes = train_ds.classes
assert classes == val_ds.classes
print("classes:", train_ds.class_to_idx, "| train", len(train_ds), "| val", len(val_ds))

BATCH = 64
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=(device == "cuda"))
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=(device == "cuda"))


## 3. Model: frozen ResNet18, new head
All pretrained weights have `requires_grad = False`; only the new final layer (512 numbers in, 3 out) is trained. The network stays in `eval()` mode throughout so the frozen BatchNorm layers keep their ImageNet statistics.

In [ ]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(classes))     # new layer: trainable by default
model = model.to(device)

EPOCHS = 12
LR = 1e-3
params = [p for p in model.parameters() if p.requires_grad]
print("trainable parameters:", sum(p.numel() for p in params), "of", sum(p.numel() for p in model.parameters()))
opt = torch.optim.Adam(params, lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()


## 4. Train
Each epoch prints train and validation loss and accuracy. The best epoch (highest validation accuracy, ties broken by lower validation loss) is saved. If the first epoch's validation accuracy is already near 100%, or stays near 33%, tell me before trusting anything.

In [ ]:
def run_epoch(loader, train):
    model.eval()                         # frozen backbone: keep BatchNorm statistics fixed
    n = correct = 0; loss_sum = 0.0
    preds, targets = [], []
    with torch.enable_grad() if train else torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            n += len(y); loss_sum += loss.item() * len(y)
            p = out.argmax(1); correct += (p == y).sum().item()
            preds.append(p.cpu()); targets.append(y.cpu())
    return loss_sum / n, correct / n, torch.cat(preds).numpy(), torch.cat(targets).numpy()

CKPT_PATH = f"{CKPT_DIR}/phase1_frozen_resnet18_best.pt"
hist, best, cm_best = [], None, None
for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta, _, _ = run_epoch(train_loader, True)
    vl, va, vp, vt = run_epoch(val_loader, False)
    sched.step()
    hist.append({"epoch": ep, "train_loss": tl, "train_acc": ta, "val_loss": vl, "val_acc": va, "sec": time.time() - t0})
    is_best = best is None or (va, -vl) > (best["val_acc"], -best["val_loss"])
    if is_best:
        best = hist[-1].copy()
        cm_best = np.zeros((len(classes), len(classes)), int)
        np.add.at(cm_best, (vt, vp), 1)
        torch.save({"model": model.state_dict(), "classes": classes, "epoch": ep, "val_acc": va, "val_loss": vl,
                    "config": {"epochs": EPOCHS, "lr": LR, "batch": BATCH, "seed": SEED}}, CKPT_PATH)
    print(f"epoch {ep:2d}/{EPOCHS}  train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f} | {time.time() - t0:.0f}s" + ("  <- best, saved" if is_best else ""))
hist = pd.DataFrame(hist)
hist.to_csv(f"{CKPT_DIR}/phase1_history.csv", index=False)


## 5. Results on the validation set

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(hist.epoch, hist.train_loss, label="train"); ax[0].plot(hist.epoch, hist.val_loss, label="val"); ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(hist.epoch, hist.train_acc, label="train"); ax[1].plot(hist.epoch, hist.val_acc, label="val"); ax[1].set_title("accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

n_val = len(val_ds)
ci = 1.96 * np.sqrt(best["val_acc"] * (1 - best["val_acc"]) / n_val)
print(f"BASELINE (frozen ResNet18): best epoch {int(best['epoch'])}, validation accuracy {best['val_acc']:.4f} (+/- {ci:.3f} at 95%), val loss {best['val_loss']:.4f}")
print(f"train accuracy at that epoch: {best['train_acc']:.4f}  (a large gap above val accuracy means overfitting)")

rec = cm_best.diagonal() / cm_best.sum(1)
prec = cm_best.diagonal() / np.maximum(cm_best.sum(0), 1)
print("\nper class (best epoch):")
print(pd.DataFrame({"recall": rec.round(3), "precision": prec.round(3), "val images": cm_best.sum(1)}, index=classes))
print("\nconfusion matrix (rows = true class, columns = predicted):")
print(pd.DataFrame(cm_best, index=[f"true {c}" for c in classes], columns=[f"pred {c}" for c in classes]))


### How to read this
- **Confidence interval:** with 850 validation images, the 95% range is roughly 2 points wide each way near 90%. A phase-2 gain smaller than that could be noise.
- **Recall per class** is the share of that class's true images the model caught. Missing a diseased plant costs more than a false alarm, so look at HEALTHY precision (diseased leaves called healthy would drag it down) and the MLN/MSV recalls.
- **Which classes get confused** is the useful part. MLN and MSV looked alike in your Day 2 pictures; if most errors are between those two, that's expected.

## 6. Save a summary
Paste `day5_summary.md` back into the chat. The checkpoint and `phase1_history.csv` are already on your Drive in `maize_ckpt`.

In [ ]:
lines = [
    "# Day 5-6 summary (frozen ResNet18 baseline)",
    f"- config: epochs {EPOCHS}, Adam lr {LR} (cosine), batch {BATCH}, seed {SEED}; train aug as Day 4; test set NOT used",
    f"- best epoch {int(best['epoch'])}: val accuracy {best['val_acc']:.4f} (+/- {ci:.3f}), val loss {best['val_loss']:.4f}, train accuracy {best['train_acc']:.4f}",
    f"- last epoch: val accuracy {hist.val_acc.iloc[-1]:.4f}, train accuracy {hist.train_acc.iloc[-1]:.4f}",
    f"- seconds per epoch: about {hist.sec.mean():.0f}",
    "- per-class recall / precision (best epoch): " + "; ".join(f"{c} {r:.3f} / {p:.3f}" for c, r, p in zip(classes, rec, prec)),
    "- confusion matrix (rows true, columns predicted, order " + ", ".join(classes) + "): " + str(cm_best.tolist()),
    f"- checkpoint: {CKPT_PATH}",
]
open("/content/day5_summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))
try:
    from google.colab import files
    files.download("/content/day5_summary.md")
except ImportError:
    pass
